# 4 — Complete all 288 Stage 3 episodes
Launch ID first. Only after ID exits and 96 ID artifacts exist, launch OOD. Never run both workers concurrently.

In [ ]:
import os, subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage3"; P=Path.home()/"LIBERO-plus"; GPU=(Path.home()/"stage3_gpu.txt").read_text().strip()
def launch(scene):
    pidfile=OUT/f"stage3_{scene}.pid"; log=OUT/f"stage3_{scene}.log"
    if pidfile.exists():
        old=int(pidfile.read_text())
        try: os.kill(old,0); raise SystemExit(f"STOP: {scene} worker {old} is alive")
        except ProcessLookupError: pass
    py=Path.home()/("venv-stage1-id/bin/python" if scene=="id" else "venv-stage1-ood/bin/python")
    env=os.environ.copy(); env.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1"})
    if scene=="ood": env["PYTHONPATH"]=str(P)
    cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage3","--config",str(R/"async_vla_benchmark/configs/stage3.yaml"),"--manifest",str(OUT/"stage3_manifest.csv"),"--output-dir",str(OUT),"--scene",scene,"--resume","--verbose"]
    fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)); print("launched",scene,proc.pid,log)
launch("id")


In [ ]:
# Re-run after reconnecting. Progress is computed from durable artifacts.
import csv
manifest=list(csv.DictReader(open(OUT/"stage3_manifest.csv"))); planned={r['run_id'] for r in manifest}; done={p.stem for p in (OUT/"episodes").glob("*.json")}; print(f"episodes {len(planned & done)}/288; remaining {288-len(planned & done)}")
for scene in ("id","ood"):
    pf=OUT/f"stage3_{scene}.pid"; alive=False
    if pf.exists():
        try: os.kill(int(pf.read_text()),0); alive=True
        except (ProcessLookupError,PermissionError): pass
    log=OUT/f"stage3_{scene}.log"; print(scene,"alive=",alive); print(''.join(log.read_text().splitlines(True)[-10:]) if log.exists() else '(not started)')
print("Checkpoint ~/stage3 off-machine regularly. --resume skips complete artifacts.")


In [ ]:
# Run only after the ID worker has exited and all 96 ID artifacts exist.
rows=list(csv.DictReader(open(OUT/"stage3_manifest.csv"))); expected={r['run_id'] for r in rows if r['scene']=='id'}; done={p.stem for p in (OUT/"episodes").glob("*.json")}
if len(expected & done)!=96: raise SystemExit(f"STOP: ID is {len(expected & done)}/96")
launch("ood")
